# 使用开源工具的综合数据生成器

## 概述
本笔记本演示了如何使用以下方法构建可用于生产的合成数据生成器：
- **Faker**：用于真实的结构化数据（姓名、电子邮件等）
- **Ollama**：使用本地法学硕士由人工智能生成上下文内容
- **Pandas**：用于数据操作和导出

## 用例
使用与职位和经验水平相匹配的人工智能驱动的传记生成真实的员工数据集。

## 第 1 步：设置和导入

首先，我们将安装所需的库并将它们导入到我们的笔记本中。

In [ ]:
# 安装所需的包
!pip install faker ollama pandas tqdm -q 

In [ ]:
# 导入库
import pandas as pd
from faker import Faker
import ollama
from tqdm import tqdm
import random
from typing import Dict, List, Any
import warnings

warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## 第 2 步：配置

在一个集中位置定义数据集架构和生成参数，以便于修改。

In [ ]:
# 配置字典
CONFIG = {
    # 发电参数
    "NUM_ROWS": 50,
    "BATCH_SIZE": 10,  # Process in batches to manage memory
    
    # 模式定义
    "SCHEMA": {
        "Name": "faker",           # Generated by Faker
        "Email": "faker",          # Generated by Faker
        "Job Title": "faker",      # Generated by Faker
        "Seniority": "custom",     # Custom logic (Junior, Mid, Senior)
        "Years of Experience": "custom",  # Custom logic (based on seniority)
        "AI Generated Bio": "llm"  # Generated by Ollama LLM
    },
    
    # 法学硕士配置
    "LLM_MODEL": "llama3",  # Options: llama3, mistral, phi, etc.
    "LLM_TEMPERATURE": 0.7,
    
    # 职位名称选项
    "JOB_TITLES": [
        "Software Engineer", "Data Scientist", "Product Manager",
        "UX Designer", "DevOps Engineer", "Marketing Manager",
        "Sales Representative", "HR Manager", "Financial Analyst"
    ],
    
    # 资历级别
    "SENIORITY_LEVELS": ["Junior", "Mid", "Senior"],
    
    # 导出配置
    "OUTPUT_FILE": "synthetic_data.csv"
}

print("Configuration loaded:")
print(f"  • Generating {CONFIG['NUM_ROWS']} rows")
print(f"  • Using LLM model: {CONFIG['LLM_MODEL']}")
print(f"  • Schema fields: {', '.join(CONFIG['SCHEMA'].keys())}")

## 步骤 3：核心生成器类

`SyntheticDataEngine` 类协调整个数据生成过程：
- 使用 **Faker** 作为标准字段（姓名、电子邮件）
- 连接到 **Ollama** 以获取人工智能生成的内容
- 实现**业务逻辑**以实现数据一致性
- 批量处理数据以提高效率

In [ ]:
class SyntheticDataEngine:
    """
    A comprehensive synthetic data generator that combines traditional fake data
    with AI-generated content using local LLMs via Ollama.
    """
    
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the data generator with configuration.
        
        Args:
            config: Configuration dictionary containing schema, LLM settings, etc.
        """
        self.config = config
        self.faker = Faker()
        Faker.seed(42)  # For reproducibility
        random.seed(42)
        
        print(f"🚀 SyntheticDataEngine initialized")
        print(f"   Model: {config['LLM_MODEL']}")
        
    def _generate_faker_field(self, field_name: str) -> str:
        """Generate data using Faker library."""
        if field_name == "Name":
            return self.faker.name()
        elif field_name == "Email":
            return self.faker.email()
        elif field_name == "Job Title":
            return random.choice(self.config["JOB_TITLES"])
        return ""
    
    def _apply_business_logic(self, row_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Apply business rules to ensure data consistency.
        
        Business Rules:
        1. Junior: 0-3 years of experience
        2. Mid: 3-7 years of experience
        3. Senior: 7+ years of experience
        """
        # 确定工龄级别
        seniority = random.choice(self.config["SENIORITY_LEVELS"])
        row_data["Seniority"] = seniority
        
        # 根据资历应用经验规则
        if seniority == "Junior":
            years_exp = random.randint(0, 3)
        elif seniority == "Mid":
            years_exp = random.randint(3, 7)
        else:  # Senior
            years_exp = random.randint(7, 15)
        
        row_data["Years of Experience"] = years_exp
        
        return row_data
    
    def _generate_ai_bio(self, job_title: str, seniority: str, years_exp: int) -> str:
        """
        Generate a contextual biography using Ollama LLM.
        
        Args:
            job_title: The person's job title
            seniority: Seniority level (Junior/Mid/Senior)
            years_exp: Years of experience
            
        Returns:
            AI-generated professional biography
        """
        prompt = f"""Write a brief professional bio (2-3 sentences) for a {seniority} {job_title} with {years_exp} years of experience. 
Make it realistic and professional. Focus on skills and achievements relevant to their role.
Do not include a name."""
        
        try:
            response = ollama.generate(
                model=self.config["LLM_MODEL"],
                prompt=prompt,
                options={
                    "temperature": self.config["LLM_TEMPERATURE"],
                    "num_predict": 100  # Limit response length
                }
            )
            return response['response'].strip()
        except Exception as e:
            # LLM失败后的回退
            return f"Experienced {seniority} {job_title} with {years_exp} years in the industry."
    
    def generate_row(self) -> Dict[str, Any]:
        """
        Generate a single row of synthetic data.
        
        Returns:
            Dictionary containing all fields for one record
        """
        row_data = {}
        
        # 第1步：生成Faker字段
        for field_name, field_type in self.config["SCHEMA"].items():
            if field_type == "faker":
                row_data[field_name] = self._generate_faker_field(field_name)
        
        # 第 2 步：将业务逻辑应用于自定义字段
        row_data = self._apply_business_logic(row_data)
        
        # 第 3 步：生成 AI 内容（法学硕士）
        if "AI Generated Bio" in self.config["SCHEMA"]:
            row_data["AI Generated Bio"] = self._generate_ai_bio(
                row_data["Job Title"],
                row_data["Seniority"],
                row_data["Years of Experience"]
            )
        
        return row_data
    
    def generate_batch(self, batch_size: int) -> List[Dict[str, Any]]:
        """
        Generate a batch of synthetic data rows.
        
        Args:
            batch_size: Number of rows to generate
            
        Returns:
            List of dictionaries, each representing a row
        """
        return [self.generate_row() for _ in range(batch_size)]
    
    def generate_dataset(self) -> pd.DataFrame:
        """
        Generate the complete dataset with progress tracking.
        
        Returns:
            Pandas DataFrame containing all synthetic data
        """
        num_rows = self.config["NUM_ROWS"]
        batch_size = self.config["BATCH_SIZE"]
        
        all_data = []
        num_batches = (num_rows + batch_size - 1) // batch_size
        
        print(f"\n📊 Generating {num_rows} rows in {num_batches} batches...")
        
        with tqdm(total=num_rows, desc="Generating data", unit="rows") as pbar:
            for batch_num in range(num_batches):
                # 计算该批次的行数
                remaining_rows = num_rows - len(all_data)
                current_batch_size = min(batch_size, remaining_rows)
                
                # 生成批次
                batch_data = self.generate_batch(current_batch_size)
                all_data.extend(batch_data)
                
                # 更新进度
                pbar.update(current_batch_size)
        
        # 转换为数据帧
        df = pd.DataFrame(all_data)
        print(f"✓ Dataset generation complete! Shape: {df.shape}")
        
        return df

print("✓ SyntheticDataEngine class defined")

## 步骤 4：生成数据集

现在我们将实例化引擎并生成带有进度条的合成数据。

In [ ]:
# 初始化引擎
engine = SyntheticDataEngine(CONFIG)

# 生成数据集
df = engine.generate_dataset()

## 步骤 5：数据验证和探索

让我们检查生成的数据以确保质量和一致性。

In [ ]:
### 5.1 预览前 5 行

print("📋 First 5 rows of generated data:\n")
df.head()

In [ ]:
### 5.2 数据集信息

print("ℹ️  Dataset Information:\n")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names and Types:")
print(df.dtypes)
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
### 5.3 统计总结

print("📊 Statistical Summary:\n")
df.describe(include='all')

### 5.4 业务逻辑验证

验证我们的业务规则是否正确应用：
- 初级员工应有0-3年经验
- 中层员工应有3-7年的工作经验
- 高级员工应有7年以上工作经验

In [ ]:
print("🔍 Business Logic Validation:\n")

# 按资历分组并显示经验统计
validation = df.groupby('Seniority')['Years of Experience'].agg(['min', 'max', 'mean', 'count'])
print(validation)

# 检查是否有违规行为
print("\n✓ Validation Results:")
junior_valid = df[df['Seniority'] == 'Junior']['Years of Experience'].max() <= 3
mid_valid = (df[df['Seniority'] == 'Mid']['Years of Experience'].min() >= 3) and \
            (df[df['Seniority'] == 'Mid']['Years of Experience'].max() <= 7)
senior_valid = df[df['Seniority'] == 'Senior']['Years of Experience'].min() >= 7

print(f"  Junior rules: {'✓ PASS' if junior_valid else '✗ FAIL'}")
print(f"  Mid rules: {'✓ PASS' if mid_valid else '✗ FAIL'}")
print(f"  Senior rules: {'✓ PASS' if senior_valid else '✗ FAIL'}")

### 5.5 AI 生成的 Bios 示例

让我们检查一些人工智能生成的传记来评估质量：

In [ ]:
print("🤖 Sample AI-Generated Biographies:\n")

# 显示 3 个不同资历级别的随机样本
for seniority in ['Junior', 'Mid', 'Senior']:
    sample = df[df['Seniority'] == seniority].sample(1).iloc[0]
    print(f"{'='*70}")
    print(f"Name: {sample['Name']}")
    print(f"Title: {sample['Seniority']} {sample['Job Title']}")
    print(f"Experience: {sample['Years of Experience']} years")
    print(f"\nBio: {sample['AI Generated Bio']}")
    print()

## 步骤 6：导出数据

将生成的数据集保存到 CSV 文件以供进一步使用。

In [ ]:
# 导出为 CSV
output_file = CONFIG["OUTPUT_FILE"]
df.to_csv(output_file, index=False)

print(f"✓ Dataset exported successfully!")
print(f"  File: {output_file}")
print(f"  Size: {len(df)} rows × {len(df.columns)} columns")

# 验证文件已创建
import os
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / 1024
    print(f"  File size: {file_size:.2f} KB")

## 总结和要点

### 我们建造了什么
该笔记本演示了一个可立即投入生产的合成数据生成器，它结合了：
1. **传统的假数据**（Faker）用于结构化字段
2. **人工智能生成的内容** (Ollama)，提供上下文真实的文本
3. **业务逻辑** 确保数据一致性和真实性
4. **批处理**以提高内存效率
5. **进度跟踪**以获取用户反馈

### 架构亮点

#### 1. 模块化设计
- 配置与逻辑分离
- 可重用的`SyntheticDataEngine`类
- 易于扩展新字段或规则

#### 2. 数据质量
- 业务规则强制逻辑一致性
- 验证检查确认规则合规性
- 人工智能生成的内容增加了真实感

#### 3. 可扩展性
- 批处理可防止内存问题
- 可以轻松扩展到数千行
- 进度条为长时间操作提供反馈

#### 4. 局部优先方法
- 使用 Ollama 保护隐私并节省成本
- 没有 API 密钥或外部依赖项
- 完全控制数据生成

### 用例
- **测试**：为应用程序生成真实的测试数据
- **训练**：为 ML 模型训练创建数据集
- **演示**：使用真实数据填充演示环境
- **隐私**：用合成替代品替换敏感生产数据

### 后续步骤
要扩展此生成器，您可以：
- 添加更多字段（电话号码、地址、部门）
- 实施更复杂的业务规则
- 添加数据质量检查（电子邮件格式验证等）
- 生成相关表（员工→项目→任务）
- 添加生成的数据分布的可视化
- 导出为多种格式（JSON、Parquet、SQL）

## 奖励：快速恢复

想要生成不同的数据集？只需修改 CONFIG 并重新运行生成单元即可！

In [ ]:
# 示例：使用不同的设置生成更大的数据集
# 取消注释并运行尝试：

# 配置[“NUM_ROWS”] = 100
# CONFIG["LLM_MODEL"] = "米斯特拉尔"
# CONFIG["OUTPUT_FILE"] = "synthetic_data_large.csv"
# 
# engine_v2 = SyntheticDataEngine(CONFIG)
# df_v2 = engine_v2.generate_dataset()
# df_v2.to_csv(CONFIG["OUTPUT_FILE"]，索引=False)
# print(f"✓ 使用 {len(df_v2)} 行创建的新数据集！")

print("💡 Tip: Uncomment the code above to generate a different dataset")